# Huan luyen Crypto: MACD, RSI

# Load data

In [1]:
import sys
sys.path.append("../Common")

import CommonBinance

In [2]:
symbol = 'ETHUSDT'
from_date = '2025-06-01'
to_date = '2025-08-28'
interval = '1d'
data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, interval)
# 2025-08-28	4506.71	4633.97	4467.63	4552.40	3.016636e+05


In [3]:
data

,Datetime,Open,High,Low,Close,Volume
0,2025-06-01,2528.06,2547.79,2468.56,2539.21,332021.7088
1,2025-06-02,2539.20,2615.00,2475.00,2607.41,468889.9737
2,2025-06-03,2607.42,2655.38,2575.00,2593.36,454383.0439
3,2025-06-04,2593.35,2679.88,2583.50,2607.68,502494.8214
4,2025-06-05,2607.68,2640.86,2391.66,2414.01,808938.0344
...,...,...,...,...,...,...
91,2025-08-31,4373.70,4498.47,4372.63,4391.83,356586.1861
92,2025-09-01,4391.83,4491.84,4210.61,4314.50,513830.0395
93,2025-09-02,4314.50,4416.78,4257.85,4326.50,524605.0512
94,2025-09-03,4326.49,4490.64,4283.23,4450.46,487091.6861


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Datetime  96 non-null     datetime64[ns]
 1   Open      96 non-null     float64       
 2   High      96 non-null     float64       
 3   Low       96 non-null     float64       
 4   Close     96 non-null     float64       
 5   Volume    96 non-null     float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 4.6 KB


In [5]:
# Import các thư viện cần thiết
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [6]:
# import numpy as np
# from itertools import combinations

# basic_features = ['Open', 'High', 'Low', 'Volume', 'Momentum', 'RSI']

# # Tạo tất cả tổ hợp có thể từ basic_features
# # Sử dụng itertools để tạo tất cả các tổ hợp của các đặc trưng
# features_combinations = []
# for r in range(1, len(basic_features) + 1):
#     for combo in combinations(basic_features, r):
#         features_combinations.append(list(combo))

# print(features_combinations)
# print(len(features_combinations))

In [7]:
import numpy as np
from itertools import combinations
import talib

# Đảm bảo cột 'Datetime' được hiểu là kiểu dữ liệu datetime và đã sắp xếp
data['Datetime'] = pd.to_datetime(data['Datetime'])
data.sort_values(by='Datetime', inplace=True)

# Thêm MACD và RSI vào DataFrame
data['MACD'], data['Signal_Line'], _ = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
data['RSI'] = talib.RSI(data['Close'], timeperiod=14)

# Loại bỏ NaN
# data.dropna(inplace=True)

data.fillna(data.mean(), inplace=True)

# Khởi tạo biến mục tiêu 'y' là giá đóng cửa
y = data['Close']

# Định nghĩa cơ bản của các đặc trưng
# 'Open', 'High', 'Low', 'Volume', 'Momentum', 'RSI'
basic_features = ['Open', 'High', 'Low', 'Volume', 'MACD', 'RSI'] # To hop cac dac trung 2^6-1 = 63

# Tạo tất cả tổ hợp có thể từ basic_features
# Sử dụng itertools để tạo tất cả các tổ hợp của các đặc trưng
features_combinations = []
for r in range(1, len(basic_features) + 1):
	for combo in combinations(basic_features, r):
		features_combinations.append(list(combo))

# Luu trữ kết quả MSE cho từng tổ hợp
mse_scores = {}

# Kiểm tra từng kết hợp đặc trưng
for features in features_combinations: # 63 lan chay
	X = data[features].copy() # Copy dữ liệu để tránh thay đổi gốc
	# Thay thế NaN bằng giá trị trung bình của cột
	X.fillna(X.mean(), inplace=True)
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
	
	model = DecisionTreeRegressor(random_state=42)
	model.fit(X_train, y_train)
	
	predictions = model.predict(X_test)
	mse = mean_squared_error(y_test, predictions)
	
	mse_scores[str(features)] = mse

print(mse_scores)
# Tìm kết hợp đặc trưng với MSE thấp nhất
best_features = min(mse_scores, key=mse_scores.get)
print(f"Best features: {best_features} with MSE: {mse_scores[best_features]}")

{"['Open']": 39747.653569999966, "['High']": 8614.428749999995, "['Low']": 15717.590764999975, "['Volume']": 1071756.560955, "['MACD']": 240661.01558695306, "['RSI']": 597495.5465662503, "['Open', 'High']": 11991.638124999992, "['Open', 'Low']": 22006.690084999987, "['Open', 'Volume']": 51549.68458499994, "['Open', 'MACD']": 48759.58517499996, "['Open', 'RSI']": 46015.23494999994, "['High', 'Low']": 12009.709189999983, "['High', 'Volume']": 9328.736704999996, "['High', 'MACD']": 13186.603614999996, "['High', 'RSI']": 7378.262169999997, "['Low', 'Volume']": 31199.646109999983, "['Low', 'MACD']": 18013.27086999999, "['Low', 'RSI']": 9239.668399999977, "['Volume', 'MACD']": 244156.51151, "['Volume', 'RSI']": 545106.3564850002, "['MACD', 'RSI']": 197692.53973000014, "['Open', 'High', 'Low']": 15970.157689999982, "['Open', 'High', 'Volume']": 16711.772889999986, "['Open', 'High', 'MACD']": 12809.225345000003, "['Open', 'High', 'RSI']": 21984.460884999982, "['Open', 'Low', 'Volume']": 32004.

# Moi lan chay la so sanh

In [8]:
# Dat MSE la gia tri vo cuc
best_mse = float('inf') # 1 so vo cuc
best_features = None
from datetime import datetime

for features in features_combinations:
	print("Bat dau huan luyen")
	print(datetime.now())
	X = data[features].copy()
	X.fillna(X.mean(), inplace=True)
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
	
	model = DecisionTreeRegressor(random_state=42)
	model.fit(X_train, y_train)
	print("Ket thuc huan luyen")
	print(datetime.now())
	predictions = model.predict(X_test)
	mse = mean_squared_error(y_test, predictions) # Tinh MSE
	
	# Kiểm tra xem MSE của lần lặp này có nhỏ hơn giá trị tốt nhất hiện tại không
	if mse < best_mse:
		best_mse = mse
		best_features = features

print(f"Best features: {best_features} with MSE: {best_mse}")

Bat dau huan luyen
2025-09-04 21:01:30.179000
Ket thuc huan luyen
2025-09-04 21:01:30.182967
Bat dau huan luyen
2025-09-04 21:01:30.184966
Ket thuc huan luyen
2025-09-04 21:01:30.188969
Bat dau huan luyen
2025-09-04 21:01:30.190991
Ket thuc huan luyen
2025-09-04 21:01:30.195962
Bat dau huan luyen
2025-09-04 21:01:30.197960
Ket thuc huan luyen
2025-09-04 21:01:30.202001
Bat dau huan luyen
2025-09-04 21:01:30.203986
Ket thuc huan luyen
2025-09-04 21:01:30.207985
Bat dau huan luyen
2025-09-04 21:01:30.209986
Ket thuc huan luyen
2025-09-04 21:01:30.213987
Bat dau huan luyen
2025-09-04 21:01:30.215983
Ket thuc huan luyen
2025-09-04 21:01:30.219945
Bat dau huan luyen
2025-09-04 21:01:30.221981
Ket thuc huan luyen
2025-09-04 21:01:30.226978
Bat dau huan luyen
2025-09-04 21:01:30.227977
Ket thuc huan luyen
2025-09-04 21:01:30.232971
Bat dau huan luyen
2025-09-04 21:01:30.234973
Ket thuc huan luyen
2025-09-04 21:01:30.239933
Bat dau huan luyen
2025-09-04 21:01:30.240943
Ket thuc huan luyen
2025

# Luu model ra 1 file

In [9]:
# 2 tuan huan luyen 1 lan => model luu ra 1 file
# model duoc load tu file